In [8]:
import tkinter as tk
from tkinter import messagebox, ttk
import csv
import os
import random

# --- File Configuration ---
DOWNLOADS_PATH = os.path.join(os.path.expanduser("~"), "Downloads")
PROP_FILE = os.path.join(DOWNLOADS_PATH, "properties.csv")

def initialize_files():
    """Create CSV if it doesn't exist."""
    if not os.path.exists(PROP_FILE):
        with open(PROP_FILE, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["seller_phone", "area", "city", "landmark", "sqft", "cost", "buyer_name"])

class PropertyApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Elite Property Hub")
        self.root.geometry("1000x700")
        self.root.configure(bg="#f0f2f5")
        
        initialize_files()
        self.current_seller = None
        self.editing_row_data = None 
        self.generated_otp = None
        self.main_menu()

    def clear_screen(self):
        for widget in self.root.winfo_children():
            widget.destroy()

    def get_unique_values(self, column_name):
        """Dynamic dropdown values based on CSV content."""
        values = set(["All"])
        try:
            with open(PROP_FILE, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row[column_name]: values.add(row[column_name])
        except: pass
        return sorted(list(values))

    # --- BUYER / MAIN SEARCH ---
    def main_menu(self):
        self.clear_screen()
        
        # Header
        header = tk.Frame(self.root, bg="#1a237e", height=70)
        header.pack(fill="x")
        tk.Label(header, text="PROPERTY MARKETPLACE", font=("Arial", 20, "bold"), fg="white", bg="#1a237e").pack(pady=15)

        # Search Bar
        s_frame = tk.Frame(self.root, bg="white", padx=15, pady=15)
        s_frame.pack(pady=10, fill="x", padx=20)

        tk.Label(s_frame, text="City:", bg="white").grid(row=0, column=0, padx=5)
        self.city_var = tk.StringVar(value="All")
        self.city_cb = ttk.Combobox(s_frame, textvariable=self.city_var, values=self.get_unique_values("city"), width=15)
        self.city_cb.grid(row=0, column=1, padx=5)

        tk.Label(s_frame, text="Area:", bg="white").grid(row=0, column=2, padx=5)
        self.area_var = tk.StringVar(value="All")
        self.area_cb = ttk.Combobox(s_frame, textvariable=self.area_var, values=self.get_unique_values("area"), width=15)
        self.area_cb.grid(row=0, column=3, padx=5)

        tk.Button(s_frame, text="SEARCH", bg="#1a237e", fg="white", width=12, command=self.load_buyer_data).grid(row=0, column=4, padx=10)
        tk.Button(s_frame, text="SELLER LOGIN", bg="#f57c00", fg="white", width=12, command=self.login_page).grid(row=0, column=5, padx=5)

        # Table
        cols = ("Area", "City", "Landmark", "SqFt", "Cost", "Contact", "Buyer")
        self.tree = ttk.Treeview(self.root, columns=cols, show='headings')
        for col in cols: 
            self.tree.heading(col, text=col)
            self.tree.column(col, width=100)
        self.tree.pack(pady=10, padx=20, fill="both", expand=True)
        
        self.load_buyer_data()

    def load_buyer_data(self):
        """Corrected Filtering Logic."""
        for i in self.tree.get_children(): self.tree.delete(i)
        city_f = self.city_var.get()
        area_f = self.area_var.get()

        try:
            with open(PROP_FILE, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    c_match = (city_f == "All" or row['city'] == city_f)
                    a_match = (area_f == "All" or row['area'] == area_f)
                    if c_match and a_match:
                        self.tree.insert("", "end", values=(row['area'], row['city'], row['landmark'], row['sqft'], row['cost'], row['seller_phone'], row['buyer_name']))
        except: pass

    # --- LOGIN SYSTEM ---
    def login_page(self):
        self.clear_screen()
        box = tk.Frame(self.root, bg="white", padx=30, pady=30, highlightthickness=1, highlightbackground="#ddd")
        box.place(relx=0.5, rely=0.5, anchor="center")

        tk.Label(box, text="Seller Login", font=("Arial", 16, "bold"), bg="white").pack(pady=10)
        self.ph_ent = tk.Entry(box, width=25, font=("Arial", 12)); self.ph_ent.pack(pady=5)
        self.ph_ent.insert(0, "Phone (10 Digits)")

        tk.Button(box, text="Get OTP", command=self.send_otp).pack(pady=5)
        self.otp_ent = tk.Entry(box, width=10, font=("Arial", 12)); self.otp_ent.pack(pady=5)
        
        tk.Button(box, text="VERIFY & ENTER", bg="#2e7d32", fg="white", font=("Arial", 10, "bold"), width=20, command=self.do_login).pack(pady=15)
        tk.Button(box, text="Back", command=self.main_menu).pack()

    def send_otp(self):
        ph = self.ph_ent.get()
        if len(ph) == 10 and ph.isdigit():
            self.generated_otp = str(random.randint(1000, 9999))
            messagebox.showinfo("OTP", f"Code: {self.generated_otp}")
        else: messagebox.showerror("Error", "Enter 10-digit number")

    def do_login(self):
        if self.otp_ent.get() == self.generated_otp and self.generated_otp:
            self.current_seller = self.ph_ent.get()
            self.seller_dashboard()
        else: messagebox.showerror("Error", "Wrong OTP")

    # --- SELLER DASHBOARD ---
    def seller_dashboard(self):
        self.clear_screen()
        # Top Bar
        bar = tk.Frame(self.root, bg="#1a237e", height=50)
        bar.pack(fill="x")
        tk.Label(bar, text=f"Dashboard: {self.current_seller}", fg="white", bg="#1a237e").pack(side="left", padx=20)
        tk.Button(bar, text="Logout", command=self.main_menu).pack(side="right", padx=20, pady=10)

        # CRUD Form
        f_frame = tk.Frame(self.root, bg="white", pady=15)
        f_frame.pack(fill="x", padx=20, pady=10)
        
        self.entries = {}
        fields = ["Area", "City", "Landmark", "SqFt", "Cost", "Buyer Name"]
        for f in fields:
            box = tk.Frame(f_frame, bg="white")
            box.pack(side="left", padx=10)
            tk.Label(box, text=f, bg="white", font=("Arial", 9)).pack()
            e = tk.Entry(box, width=12)
            e.pack()
            self.entries[f] = e

        # Buttons
        b_frame = tk.Frame(self.root)
        b_frame.pack(pady=10)
        self.btn_save = tk.Button(b_frame, text="ADD LISTING", bg="#2e7d32", fg="white", width=15, command=self.save_logic)
        self.btn_save.pack(side="left", padx=5)
        tk.Button(b_frame, text="EDIT SELECTED", bg="#1565c0", fg="white", width=15, command=self.edit_logic).pack(side="left", padx=5)
        tk.Button(b_frame, text="DELETE", bg="#c62828", fg="white", width=15, command=self.delete_logic).pack(side="left", padx=5)

        # Seller Table
        self.s_tree = ttk.Treeview(self.root, columns=fields, show='headings')
        for f in fields: self.s_tree.heading(f, text=f)
        self.s_tree.pack(fill="both", expand=True, padx=20, pady=10)
        
        self.refresh_seller_data()

    def refresh_seller_data(self):
        for i in self.s_tree.get_children(): self.s_tree.delete(i)
        with open(PROP_FILE, 'r') as f:
            for row in csv.DictReader(f):
                if row['seller_phone'] == self.current_seller:
                    self.s_tree.insert("", "end", values=(row['area'], row['city'], row['landmark'], row['sqft'], row['cost'], row['buyer_name']))

    def edit_logic(self):
        sel = self.s_tree.selection()
        if not sel: 
            messagebox.showwarning("Select", "Select a row to edit")
            return
        vals = self.s_tree.item(sel)['values']
        field_list = ["Area", "City", "Landmark", "SqFt", "Cost", "Buyer Name"]
        for i, f in enumerate(field_list):
            self.entries[f].delete(0, tk.END)
            self.entries[f].insert(0, vals[i])
        # Store as string list for matching later
        self.editing_row_data = [str(v) for v in vals]
        self.btn_save.config(text="UPDATE LISTING", bg="#f57c00")

    def save_logic(self):
        new_row = [self.current_seller] + [self.entries[f].get() for f in ["Area", "City", "Landmark", "SqFt", "Cost", "Buyer Name"]]
        
        if "" in new_row:
            messagebox.showwarning("Error", "Fill all fields")
            return

        updated_list = []
        found = False
        with open(PROP_FILE, 'r') as f:
            reader = csv.reader(f)
            header = next(reader)
            updated_list.append(header)
            for row in reader:
                # Check if we are updating this specific row
                if self.editing_row_data and row[0] == self.current_seller and row[1:] == self.editing_row_data:
                    updated_list.append(new_row)
                    found = True
                else:
                    updated_list.append(row)
        
        if not found: updated_list.append(new_row)

        with open(PROP_FILE, 'w', newline='') as f:
            csv.writer(f).writerows(updated_list)
        
        self.editing_row_data = None
        self.btn_save.config(text="ADD LISTING", bg="#2e7d32")
        self.refresh_seller_data()
        for e in self.entries.values(): e.delete(0, tk.END)

    def delete_logic(self):
        sel = self.s_tree.selection()
        if not sel: return
        vals = [str(v) for v in self.s_tree.item(sel)['values']]
        
        updated_list = []
        with open(PROP_FILE, 'r') as f:
            reader = csv.reader(f)
            updated_list.append(next(reader))
            for row in reader:
                # Only skip (delete) the exact match for this seller
                if row[0] == self.current_seller and row[1:] == vals:
                    continue
                updated_list.append(row)
                
        with open(PROP_FILE, 'w', newline='') as f:
            csv.writer(f).writerows(updated_list)
        self.refresh_seller_data()

if __name__ == "__main__":
    root = tk.Tk()
    app = PropertyApp(root)
    root.mainloop()
